In [1]:
import numpy as np
import tensorflow as tf

np.random.seed(42)

tf.random.set_seed(42)

In [2]:
import pandas as pd
import time
import os

In [3]:
DATASET_COLUMNS = ["target", "ids", "date", "flag", "user", "text"]
DATASET_ENCODING = "ISO-8859-1"

In [4]:
for dirname, _, filenames in os.walk('/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))
df = pd.read_csv(r"C:\Users\anpur\Downloads\training.1600000.processed.noemoticon.csv\training.1600000.processed.noemoticon.csv", encoding=DATASET_ENCODING, names=DATASET_COLUMNS)

In [5]:
df.tail()

,target,ids,date,flag,user,text
1599995,4,2193601966,Tue Jun 16 08:40:49 PDT 2009,NO_QUERY,AmandaMarie1028,Just woke up. Having no school is the best fee...
1599996,4,2193601969,Tue Jun 16 08:40:49 PDT 2009,NO_QUERY,TheWDBoards,TheWDB.com - Very cool to hear old Walt interv...
1599997,4,2193601991,Tue Jun 16 08:40:49 PDT 2009,NO_QUERY,bpbabe,Are you ready for your MoJo Makeover? Ask me f...
1599998,4,2193602064,Tue Jun 16 08:40:49 PDT 2009,NO_QUERY,tinydiamondz,Happy 38th Birthday to my boo of alll time!!! ...
1599999,4,2193602129,Tue Jun 16 08:40:50 PDT 2009,NO_QUERY,RyanTrevMorris,happy #charitytuesday @theNSPCC @SparksCharity...


In [6]:
# Map target to string
decode_map = {0: "NEGATIVE", 2: "NEUTRAL", 4: "POSITIVE"}
def decode_sentiment(label):
    return decode_map[int(label)]
df.target = df.target.apply(lambda x: decode_sentiment(x))

In [7]:
df.head()

,target,ids,date,flag,user,text
0,NEGATIVE,1467810369,Mon Apr 06 22:19:45 PDT 2009,NO_QUERY,_TheSpecialOne_,"@switchfoot http://twitpic.com/2y1zl - Awww, t..."
1,NEGATIVE,1467810672,Mon Apr 06 22:19:49 PDT 2009,NO_QUERY,scotthamilton,is upset that he can't update his Facebook by ...
2,NEGATIVE,1467810917,Mon Apr 06 22:19:53 PDT 2009,NO_QUERY,mattycus,@Kenichan I dived many times for the ball. Man...
3,NEGATIVE,1467811184,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,ElleCTF,my whole body feels itchy and like its on fire
4,NEGATIVE,1467811193,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,Karoli,"@nationwideclass no, it's not behaving at all...."


In [8]:
import nltk
import nltk.corpus
from nltk.corpus import stopwords
nltk.download("stopwords")
stop_words = set(stopwords.words("english"))

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\anpur\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [9]:
import re
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

In [10]:
# Data Cleaning and Preprocessing
def preprocess_text(text):
    text = re.sub(r"@\S+|https?:\S+|http?:\S|[^A-Za-z0-9]+", ' ', text)
    return text.lower().strip()

df['text'] = df['text'].apply(preprocess_text)

train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

# Tokenization and Padding
max_words = 10000  
tokenizer = Tokenizer(num_words=max_words, oov_token='<OOV>')
tokenizer.fit_on_texts(train_df['text'])

train_sequences = tokenizer.texts_to_sequences(train_df['text'])
test_sequences = tokenizer.texts_to_sequences(test_df['text'])

max_length = 50 
train_padded = pad_sequences(train_sequences, maxlen=max_length, padding='post', truncating='post')
test_padded = pad_sequences(test_sequences, maxlen=max_length, padding='post', truncating='post')

# Model Definition
embedding_dim = 32 
model = Sequential([
    Embedding(input_dim=max_words, output_dim=embedding_dim, input_length=max_length),
    LSTM(64, return_sequences=True),
    LSTM(64),
    Dense(1, activation='sigmoid')
])

model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

# Model Training


X_train, y_train = train_padded, (train_df['target'] == 'POSITIVE').astype(int)
X_test, y_test = test_padded, (test_df['target'] == 'POSITIVE').astype(int)

model.fit(X_train, y_train, epochs=5, validation_data=(X_test, y_test))

#Evaluate the Model
test_loss, test_acc = model.evaluate(X_test, y_test)
print(f"Test Accuracy: {test_acc}")

# the following lines will only be for testing 

d:\Jupyter\.venv\Lib\site-packages\keras\src\layers\core\embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Epoch 1/5
40000/40000 ━━━━━━━━━━━━━━━━━━━━ 835s 21ms/step - accuracy: 0.6621 - loss: 0.5698 - val_accuracy: 0.8204 - val_loss: 0.3960
Epoch 2/5
40000/40000 ━━━━━━━━━━━━━━━━━━━━ 871s 22ms/step - accuracy: 0.8236 - loss: 0.3890 - val_accuracy: 0.8264 - val_loss: 0.3840
Epoch 3/5
40000/40000 ━━━━━━━━━━━━━━━━━━━━ 837s 21ms/step - accuracy: 0.8336 - loss: 0.3701 - val_accuracy: 0.8281 - val_loss: 0.3819
Epoch 4/5
40000/40000 ━━━━━━━━━━━━━━━━━━━━ 851s 21ms/step - accuracy: 0.8411 - loss: 0.3569 - val_accuracy: 0.8264 - val_loss: 0.3849
Epoch 5/5
40000/40000 ━━━━━━━━━━━━━━━━━━━━ 842s 21ms/step - accuracy: 0.8472 - loss: 0.3453 - val_accuracy: 0.8242 - val_loss: 0.3913
10000/10000 ━━━━━━━━━━━━━━━━━━━━ 72s 7ms/step - accuracy: 0.8238 - loss: 0.3912
Test Accuracy: 0.8241843581199646


In [11]:
train_df['text']
train_df[['text']].to_csv('token.csv',index = False)

In [12]:
model.save("text.h5")

In [13]:
model.save("text.keras")

In [15]:
from tensorflow.keras.models import load_model

# Load model (avoid compiling during load)
model = load_model(r"D:\Jupyter\Major Project\emotion_model_v2.h5", compile=False)

# Then compile it manually
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])


In [6]:
import numpy as np
from tensorflow.keras.models import load_model
from PIL import Image

# Load your trained model
model = load_model(r"D:\Jupyter\Major Project\emotion_model.h5")  

# Use hardcoded image path
image_path = r"D:\Jupyter\Major Project\train\angry\Training_43697890.jpg"

# Load and preprocess the image
img = Image.open(image_path).convert('L')  # Convert to grayscale
img = img.resize((48, 48))                 # Resize to model input
img_array = np.array(img) / 255.0          # Normalize pixel values
img_array = img_array.reshape(1, 48, 48, 1)  # Reshape to match model input

# Predict
pred = model.predict(img_array)
print("Raw probabilities:", pred)

# Class prediction
classes = ['Angry', 'Happy', 'Surprise']
predicted_class = classes[np.argmax(pred)]
print("Predicted Emotion:", predicted_class)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 191ms/step
Raw probabilities: [[0.95818263 0.04051528 0.00130221]]
Predicted Emotion: Angry


In [4]:
from tensorflow.keras.models import load_model
import numpy as np
from PIL import Image
import os
from textblob import TextBlob

# Load the image model once
model_path = r"D:\Jupyter\Major Project\emotion_model.h5"
model = load_model(model_path, compile=False)
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# Image check
def is_image_file(file_path):
    return file_path.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.gif'))

# Predict image emotion
def predict_image(image_path, image_model):
    img = Image.open(image_path).convert('L')  # Convert to grayscale
    img = img.resize((48, 48))  # Resize to match model input
    img_array = np.array(img) / 255.0
    img_array = img_array.reshape(1, 48, 48, 1)

    pred = image_model.predict(img_array)
    classes = ['Angry', 'Happy', 'Surprise']
    predicted_class = classes[np.argmax(pred)]
    print(f" Image Prediction: {predicted_class}")

# Predict sentiment from text
def predict_text_sentiment(file_path):
    with open(file_path, 'r', encoding='utf-8') as file:
        text = file.read()
    blob = TextBlob(text)
    sentiment = blob.sentiment.polarity

    if sentiment > 0:
        print(" Text Sentiment: Positive ")
    elif sentiment < 0:
        print(" Text Sentiment: Negative ")
    else:
        print(" Text Sentiment: Neutral ")

# Main input
file_path = input(" Enter path to a text file or image: ")

if os.path.isfile(file_path):
    if is_image_file(file_path):
        predict_image(file_path, model)
    elif file_path.lower().endswith('.txt'):
        predict_text_sentiment(file_path)
    else:
        print(" Unsupported file format. Please provide an image or a .txt file.")
else:
    print(" File does not exist.")


 Text Sentiment: Neutral 
